# Verbose PPO Training Loop

This notebook exposes the training loop directly. It prints every update, writes JSONL metrics, writes TensorBoard scalars, evaluates periodically, and saves checkpoints.


In [ ]:
from pathlib import Path
import subprocess
import sys

def add_repo_root_to_path(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'learning_backend' / '__init__.py').exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError('Could not find repo root containing learning_backend/__init__.py')

repo_root = add_repo_root_to_path()
requirements = repo_root / 'learning_backend' / 'requirements.txt'
try:
    import chess  # noqa: F401
    import numpy  # noqa: F401
    import tensorboard  # noqa: F401
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)])

print(f'Using Python: {sys.executable}')
print(f'Repo root: {repo_root}')


In [ ]:
from dataclasses import asdict
import json
from pathlib import Path

import numpy as np

from learning_backend.envs.chess960_env import Chess960Env
from learning_backend.experiments.tensorboard_logging import TensorBoardLogger
from learning_backend.rl.ppo import (
    PPO_PRESETS,
    PPOConfig,
    MaskedLinearPolicyValue,
    collect_rollout,
    update_policy,
    evaluate_ppo_model,
    save_ppo_checkpoint,
    format_update_metrics,
)


## Configuration
Adjust these values before running the loop.


In [ ]:
preset = 'ppo_debug_fast'
updates = 5
seed = 42
eval_every = 1
eval_games = 4
eval_baseline = 'random'
run_dir = repo_root / 'learning_backend' / 'experiments' / 'runs' / 'notebook_verbose_ppo'
tensorboard_dir = run_dir / 'tensorboard'
metrics_path = run_dir / 'training_log.jsonl'
checkpoint_dir = run_dir / 'checkpoints'

config = PPOConfig(**{**asdict(PPO_PRESETS[preset]), 'seed': seed})
run_dir.mkdir(parents=True, exist_ok=True)
checkpoint_dir.mkdir(parents=True, exist_ok=True)
print(config)
print(f'Run dir: {run_dir}')
print(f'TensorBoard dir: {tensorboard_dir}')


## Initialize Environment And Model


In [ ]:
rng = np.random.default_rng(seed)
env = Chess960Env(seed=seed, reward_debug=True)
observation, info = env.reset(seed=seed)
model = MaskedLinearPolicyValue(
    obs_size=int(np.prod(observation.shape)),
    action_size=len(info['legal_action_mask']),
    rng=rng,
)

print('observation shape:', observation.shape)
print('legal actions:', len(info['legal_actions']))
print('action size:', model.action_size)
print('policy_w shape:', model.policy_w.shape)
print('value_w shape:', model.value_w.shape)


## Manual Training Loop
Run this cell to see every update printed and written to TensorBoard.


In [ ]:
all_metrics = []
with TensorBoardLogger(tensorboard_dir) as tb:
    for update in range(1, updates + 1):
        print(f'\n=== update {update} ===')
        batch, rollout_metrics = collect_rollout(env, model, config, rng=rng)
        print('rollout:', json.dumps(rollout_metrics, indent=2))
        print('batch rewards:', {
            'sum': float(batch.rewards.sum()),
            'mean': float(batch.rewards.mean()),
            'std': float(batch.rewards.std()),
            'min': float(batch.rewards.min()),
            'max': float(batch.rewards.max()),
        })
        print('batch advantages:', {
            'mean': float(batch.advantages.mean()),
            'std': float(batch.advantages.std()),
            'min': float(batch.advantages.min()),
            'max': float(batch.advantages.max()),
        })

        train_metrics = update_policy(model, batch, config, rng=rng)
        row = {
            'update': update,
            **rollout_metrics,
            **train_metrics,
            'reward': float(batch.rewards.sum()),
            'illegal_move_count': int(rollout_metrics['illegal_moves']),
        }

        if eval_every and update % eval_every == 0:
            eval_metrics = evaluate_ppo_model(model, baseline=eval_baseline, games=eval_games, seed=seed + update * 1000)
            print('eval:', json.dumps(eval_metrics, indent=2))
            row.update({f'eval_{key}': value for key, value in eval_metrics.items()})
            tb.scalars(f'eval/{eval_baseline}', eval_metrics, update)

        tb.scalars('train', row, update)
        tb.flush()
        all_metrics.append(row)
        with metrics_path.open('a') as handle:
            handle.write(json.dumps(row) + '\n')
        checkpoint_path = save_ppo_checkpoint(
            model,
            config,
            checkpoint_dir / f'ppo-update-{update:06d}.json',
            extra={'update': update, 'metrics_path': str(metrics_path), 'metrics': row},
        )
        print(format_update_metrics(row))
        print('checkpoint:', checkpoint_path)

print('finished')


## TensorBoard
Run this cell in Jupyter to open TensorBoard inline, or run the shell command in a terminal.


In [ ]:
print(f'TensorBoard logdir: {tensorboard_dir}')
print(f'Terminal command: tensorboard --logdir {tensorboard_dir} --port 6006')
try:
    get_ipython().run_line_magic('load_ext', 'tensorboard')
    get_ipython().run_line_magic('tensorboard', f'--logdir {tensorboard_dir} --port 6006')
except NameError:
    pass


## Inspect Metrics


In [ ]:
for row in all_metrics:
    print(format_update_metrics(row))
